# From a code on paper to a complete qodec

A quantum error correcting code, as it appears in a paper, is a short list of
Pauli operators: the stabilizers that define the codespace, and the operators
that represent the logical qubits. That is enough to reason about the code, and
nowhere near enough to describe how to prepare, preserve, or read out an encoded
state. Those circuits have to be written, checked, and kept in sync with the code.

`qdk.ec.qodec_from_code` does that step for you. Hand it a `qodec.Code` and it
returns a complete, verified [qodec](https://github.com/microsoft/qodec): a
logical instruction set over the code's logical qubits, lowering to physical
stim operations, with a synthesized circuit behind every instruction.

This notebook takes the Steane code from its stabilizers to a complete qodec
without writing a circuit by hand.

## Installing

```bash
pip install "qdk[ec]"
```

## 1. The code, as you would write it down

The Steane [[7,1,3]] code: seven physical qubits, one logical qubit, distance 3.
Six stabilizer generators — three X-type, three Z-type — and one logical X / Z
pair. This is the whole input.

In [ ]:
import qodec as qc

steane = qc.Code(
    "steane",
    stabilizers=[
        "X_0 X_3 X_4 X_6",
        "X_1 X_3 X_5 X_6",
        "X_2 X_4 X_5 X_6",
        "Z_0 Z_3 Z_4 Z_6",
        "Z_1 Z_3 Z_5 Z_6",
        "Z_2 Z_4 Z_5 Z_6",
    ],
    x=["X_0 X_1 X_3"],
    z=["Z_1 Z_2 Z_5"],
)

print(f"{len(list(steane.stabilizers))} stabilizers, {len(list(steane.x))} logical qubit(s)")


## 2. Synthesis

One call turns that into a runnable qodec.

In [ ]:
import qdk.ec as ec
from qdk.ec import action, distance, lint
from qdk.ec import qodec_from_code, synthesis_notes

qodec = qodec_from_code(steane)
print(qodec.summary())

The result is a two-layer qodec. The top layer is a *synthesized* logical ISA —
instructions that talk about the logical qubit, not the seven physical ones —
and the bottom layer is the physical stim ISA the gadgets lower into.

In [ ]:
logical = qodec.layers[0]

for mnemonic, instruction in sorted(logical.isa.instructions.items()):
    print(f"{mnemonic:12s} {instruction.description}")


## 3. The circuits it wrote

`idle` is a syndrome-extraction round: one ancilla per stabilizer, each prepared
in |+>, coupled to its stabilizer's support with a controlled Pauli, then
rotated back and measured.

Note that `CX` is used where the stabilizer has an X, and `CZ` where it has a Z.
That one uniform construction handles CSS and non-CSS codes alike, and no data
qubit is ever touched by a basis-changing gate.

In [ ]:
print(logical.gadgets["idle"].circuit.source)

Readout is transversal, and the logical Pauli gadgets are just the code's own
logical operators applied gate by gate.

In [ ]:
for mnemonic in ("prepare_z", "measure_z", "measure_x", "x0", "z0"):
    source = logical.gadgets[mnemonic].circuit.source.strip().replace("\n", " ; ")
    print(f"{mnemonic:12s} {source[:78]}")

## 4. What makes it trustworthy

Synthesis does not assert that its circuits are right — it *proves* it, twice
over, and keeps only what passes.

First, checks and readouts are never hand-derived. Each circuit is emitted as a
draft and `complete_gadget` discovers, by exact simulation, which parities of
measurement outcomes are deterministic (the checks a decoder consumes) and which
carry the logical answer (the readouts).

In [ ]:
idle = logical.gadgets["idle"]

print(f"{len(idle.checks)} checks discovered for `idle`; the first two:")
for check in list(idle.checks)[:2]:
    print("   ", [str(atom) for atom in check])

Second, every finished gadget is checked against the instruction it claims to
implement: the action its circuit *realizes* must equal the action the
instruction *declares*. Anything that fails is dropped rather than shipped, so a
gadget that survives is one whose circuit provably does what it says.

(Correctness is necessary but not sufficient — a circuit can implement the right
operation and still squander the code's protection. Section 5 measures that.)

In [ ]:
mismatches = {
    mnemonic: action.gadget_action_mismatch(gadget)
    for mnemonic, gadget in logical.gadgets.items()
    if action.gadget_action_mismatch(gadget) is not None
}
print("gadgets whose circuit disagrees with its declared action:", mismatches or "none")

The code's distance survives the trip, and the full audit runs over the
synthesized qodec exactly as it would over a hand-authored one.

In [ ]:
distance, witness = distance.code_distance_of(qodec.codes["steane"])
print("code distance:", distance, "| witness:", [str(p) for p in witness])

report = lint.diagnose(qodec)
print(f"audit: {len(report.errors())} error(s), {len(report.warnings())} warning(s)")
for diagnostic in report.errors():
    print("   ", diagnostic.rule, "|", diagnostic.summary)


> **A note on that error.** The `gadget/readout-mismatch` rule misfires on
> X-basis destructive measurement gadgets: it also fires on the hand-authored
> `c4` qodec that ships with `qdk.ec`, and it fires asymmetrically on `measure_x`
> but not `measure_z` for codes like Steane that are perfectly X/Z symmetric. It
> is a property of that audit rule, not of the synthesized circuit. The
> declared-vs-realized action check above passes for every gadget.

## 5. Serializing it

The synthesized qodec is ordinary data — it serializes, round-trips, and is the
artifact you hand to a compilation pipeline. Nothing about it is second-class
compared to a hand-written one.

In [ ]:
text = ec.to_yaml(qodec)
restored = ec.from_yaml(text)

print(f"{len(text.splitlines())} lines of YAML")
print("round-trips:", sorted(restored.layers[0].gadgets) == sorted(logical.gadgets))


## 6. When synthesis cannot finish the job

Not every instruction exists for every code, and `qodec_from_code` will not
pretend otherwise. Take the five-qubit code as it is conventionally written,
with a logical Z that carries X components.

In [ ]:
FIVE_QUBIT_STABILIZERS = [
    "Z_0 X_1 X_2 Z_3",
    "Z_1 X_2 X_3 Z_4",
    "Z_0 Z_2 X_3 X_4",
    "X_0 Z_1 Z_3 X_4",
]

as_written = qc.Code(
    "five_qubit",
    stabilizers=list(FIVE_QUBIT_STABILIZERS),
    x=["X_0 X_1 X_2 X_3 X_4"],
    z=["X_0 X_3 Z_4"],
)

partial = qodec_from_code(as_written)
print("synthesized:", sorted(partial.layers[0].gadgets))
for mnemonic, reason in synthesis_notes(partial)["omitted"].items():
    print(f"  omitted {mnemonic:12s} {reason[:88]}")


`prepare_z` resets the data qubits to |0...0> and projects into the codespace,
which pins the logical state only when the code's logical Z is a Z-type
operator. Here it is not, so no such gadget exists — and rather than emit a
circuit that quietly prepares the wrong state, synthesis omits it and says why.

The qodec it does return is still coherent: it only advertises instructions it
can actually lower.

In [ ]:
print("instructions:", sorted(partial.layers[0].isa.instructions))
print("gadgets:     ", sorted(partial.layers[0].gadgets))

Strikingly, the omission is a property of *how the code was written down*, not
of the code itself. The same five-qubit code with an all-Z logical Z — an
equally valid choice from the same coset — synthesizes more of the menu.

In [ ]:
all_z = qc.Code(
    "five_qubit_all_z",
    stabilizers=list(FIVE_QUBIT_STABILIZERS),
    x=["X_0 X_1 X_2 X_3 X_4"],
    z=["Z_0 Z_1 Z_2 Z_3 Z_4"],
)

better = qodec_from_code(all_z)
print("as written :", sorted(partial.layers[0].gadgets))
print("all-Z basis:", sorted(better.layers[0].gadgets))


Pass `strict=True` when a partial qodec is not acceptable and you would rather
be told immediately.

In [ ]:
try:
    qodec_from_code(as_written, strict=True)
except ValueError as error:
    print("strict=True raised:", str(error)[:120])

## Where to go next

* `qodec_from_code(code, flags=..., strict=...)` synthesizes a qodec.
* `synthesis_notes(qodec)` records what was built, what was omitted and why,
  and how many flag qubits were used.
* `ec.memory_program(qodec, rounds=...)` constructs the standard logical memory
  experiment.
* `qdk.ec.complete_gadget` and `qdk.ec.complete_qodec` finish hand-written
  drafts the same way synthesis finishes generated ones.
* `qdk.ec.action`, `.checks`, `.distance`, and `.lint` characterize and verify
  the result.

### Further reading

* Dennis, Kitaev, Landahl, Preskill, *Topological quantum memory*,
  quant-ph/0110143 discusses hook errors.
* Chao & Reichardt, *Quantum error correction with only two extra qubits*,
  arXiv:1705.02329 describes the flag construction for distance-3 codes.
* Chamberland & Beverland, *Flag fault-tolerant error correction with arbitrary
  distance codes*, arXiv:1708.02246 generalizes the construction.

See `qdk_ec_walkthrough.ipynb` for the authoring, profiling, and testing
lifecycle on a hand-authored qodec.